# 03 - Profile reconstruction from Gaussian-mixture parameters

This notebook confirms that elevation profiles are rebuilt from `(amplitude, mean, sigma)` triplets exactly as the pipeline expects. We compare `GaussianReconstructor.components` and `reconstruct_batch` against a closed-form Gaussian sum on hand-chosen parameters with known peaks, and check `GaussianClamp` constrains parameters into physically valid ranges.

Modules exercised: `tools.data.gaussians` (`GaussianReconstructor`, `GaussianMixture`, `GaussianClamp`).

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")

import matplotlib.pyplot as plt
import numpy             as np
import torch

NP_SEED    = 0
TORCH_SEED = 0
np.random.seed(NP_SEED)
torch.manual_seed(TORCH_SEED)
torch.use_deterministic_algorithms(False)

plt.rcParams["figure.dpi"]  = 110
plt.rcParams["figure.figsize"] = (6.0, 4.0)
plt.rcParams["axes.grid"]   = True
plt.rcParams["grid.alpha"]  = 0.3

print("Repository root :", REPO_ROOT)
print("Torch device    : cpu (CPU only, no checkpoints required)")


## Hand-chosen mixture with known peaks

We define an elevation axis and a two-component mixture: a tall narrow peak at `mu = -4` and a shorter wide peak at `mu = +6`. The reconstructed total profile must peak near these means with the prescribed amplitudes.

In [ ]:
from tools.data.gaussians import GaussianReconstructor

x_axis = np.linspace(-15.0, 15.0, 121).astype(np.float64)

params = np.array([
    1.0, -4.0, 1.5,   # g1: amp, mu, sigma
    0.6,  6.0, 3.0,   # g2: amp, mu, sigma
], dtype=np.float64)
K = 2

components = GaussianReconstructor.components(params, x_axis, K)
total      = np.sum(components, axis=0)

for k, comp in enumerate(components):
    peak_loc = x_axis[np.argmax(comp)]
    print(f'g{k+1}: peak amplitude {comp.max():.3f} at x = {peak_loc:+.2f}')


## Cross-check against a closed-form Gaussian sum

We recompute the mixture with an independent NumPy expression `a * exp(-(x-mu)^2 / (2 sigma^2))` and confirm it matches the reconstructor output to numerical tolerance (the reconstructor adds a tiny `1e-8` to the denominator, so we expect agreement, not bit equality).

In [ ]:
def closed_form(p, x, k):
    out = np.zeros_like(x)
    for i in range(k):
        a, mu, sig = p[3 * i], p[3 * i + 1], p[3 * i + 2]
        out += a * np.exp(-((x - mu) ** 2) / (2.0 * sig * sig))
    return out

reference = closed_form(params, x_axis, K)
max_abs   = float(np.max(np.abs(reference - total)))
print('max |reconstructor - closed form| =', max_abs)
assert max_abs < 1e-6


## Batched reconstruction matches per-pixel reconstruction

The predictor reconstructs profiles for whole patches with `reconstruct_batch`, which takes parameters shaped `(B, K, 3, H, W)` and an axis shaped `(1, 1, n_elev, 1, 1)`. We tile our two-Gaussian pixel across a small patch and confirm every pixel reproduces the single-pixel profile.

In [ ]:
B, H, W = 1, 3, 3
x5 = x_axis.reshape(1, 1, -1, 1, 1).astype(np.float32)

pg = np.zeros((B, K, 3, H, W), dtype=np.float32)
for k in range(K):
    pg[:, k, 0] = params[3 * k + 0]
    pg[:, k, 1] = params[3 * k + 1]
    pg[:, k, 2] = params[3 * k + 2]

batch_curves = GaussianReconstructor.reconstruct_batch(pg, x5)
print('batch_curves shape:', batch_curves.shape, '(B, n_elev, H, W)')

per_pixel = batch_curves[0, :, 0, 0]
spread    = float(np.max(np.std(batch_curves[0], axis=(1, 2))))
print('max per-elevation spread across pixels:', spread)
print('batch vs single-pixel max diff       :', float(np.max(np.abs(per_pixel - total.astype(np.float32)))))


## Plot components and total profile

The individual Gaussian components are drawn alongside the summed profile, with vertical guides at the prescribed means.

In [ ]:
fig, ax = plt.subplots(figsize=(7.0, 4.2))
colors = ['C1', 'C2']
for k, comp in enumerate(components):
    ax.plot(x_axis, comp, color=colors[k], lw=1.2, label=f'g{k+1} component')
    ax.axvline(params[3 * k + 1], color=colors[k], ls=':', lw=0.8)

ax.plot(x_axis, total, color='black', lw=1.8, label='total mixture')
ax.set_xlabel('elevation [m]')
ax.set_ylabel('intensity')
ax.set_title('Two-component Gaussian mixture reconstruction')
ax.legend()
fig.tight_layout()
plt.show()


## GaussianClamp constrains parameters to valid ranges

`GaussianClamp.apply` clamps amplitudes to `[0, amp_max]`, means to the axis range, and sigmas to `[half-step, half-range]`. We feed deliberately out-of-range parameters (negative amplitude, mean off the axis, tiny sigma) and verify each is pulled back into bounds.

In [ ]:
from tools.data.gaussians import GaussianClamp

x_t   = torch.from_numpy(x_axis.astype(np.float32))
x_min, x_max = float(x_t.min()), float(x_t.max())
x_step       = (x_max - x_min) / (x_t.shape[0] - 1)
amp_max      = 2.0

bad = torch.tensor([[
    -0.5, -99.0, 0.0,    # g1: neg amp, mean far below axis, zero sigma
     5.0,  99.0, 1e3,    # g2: amp above max, mean far above axis, huge sigma
]], dtype=torch.float32).reshape(1, 6, 1, 1)

clamped = GaussianClamp.apply(bad, x_axis=x_t, amp_max=amp_max, ppg=3).reshape(6)

print(f'amp   in [0, {amp_max}]            :', float(clamped[0]), float(clamped[3]))
print(f'mean  in [{x_min}, {x_max}]        :', float(clamped[1]), float(clamped[4]))
print(f'sigma in [{x_step*0.5:.3f}, {(x_max-x_min)*0.5}] :', float(clamped[2]), float(clamped[5]))


## Expected visual outcome

The reconstructor must agree with the closed-form sum below `1e-6`, the batched reconstruction must be pixel-constant (zero spread) and equal to the single profile, the plotted total should peak at `x = -4` (tall) and `x = +6` (short), and every clamped parameter should sit inside its printed bounds.